# Optimized Attack-Type Classification with Optuna

Standalone Kaggle-friendly notebook for optimizing the attack-type classifier on the aggregated diagnostic table.

The goal is honest generalization, not just a high cross-validation number. The primary model-selection score is leave-one-dataset-out balanced accuracy with Cohen kappa as a tie-breaker.

## 1. Imports and Environment

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

try:
    import optuna
except ImportError as exc:
    raise ImportError("Install optuna before running this notebook: pip install optuna") from exc

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError("Install xgboost before running this notebook: pip install xgboost") from exc

try:
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise ImportError("Install catboost before running this notebook: pip install catboost") from exc

try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE
    from imblearn.combine import SMOTEENN
except ImportError as exc:
    raise ImportError("Install imbalanced-learn before running this notebook: pip install imbalanced-learn") from exc

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
KAGGLE_INPUT_DIR = Path("/kaggle/input/datasets/hongthnguyn/attack-data/data")

RANDOM_STATE = 123
N_TRIALS_PER_FAMILY = 40
MAX_CV_SPLITS = 10
PRIMARY_SCENARIO = "one-data-set-out"
SELECTION_MODE = "combined"
DATASET_SCORE_WEIGHT = 0.65
MODEL_SCORE_WEIGHT = 0.35
USE_CONSTRAINED_PREDICTIONS = True
WINDOWED_AGGREGATION_ENABLED = True
WINDOWS_PER_GROUP = 5
MIN_WINDOW_ROWS = 20
TARGET_FILE = "attr_attacks_type_agr_nn.csv"
EMBEDDED_ATTACK_TYPE_TABLE_JSON = "[{\"dataset\":\"banknote\",\"model\":\"lin\",\"attack\":\"hsj\",\"bacc_test\":0.037608486,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":23.1454545455,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":15.0,\"neighborhood_size_q50\":26.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.9812670325,\"uncertainty_q0\":0.9292543271,\"uncertainty_q25\":0.9732456454,\"uncertainty_q50\":0.982738893,\"uncertainty_q75\":0.9913716392,\"uncertainty_q1\":0.999999795,\"uncertainty_minmax\":0.0707454678,\"target_approx_consistency_in_neighborhood_mean\":0.5355606133,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.3827838828,\"target_approx_consistency_in_neighborhood_q50\":0.5,\"target_approx_consistency_in_neighborhood_q75\":0.6511156187,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.4753504507,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.3509852217,\"pred_targets_consistency_in_neighborhood_q50\":0.5,\"pred_targets_consistency_in_neighborhood_q75\":0.6153846154,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.5366322588,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.40625,\"target_targets_consistency_in_neighborhood_q50\":0.5,\"target_targets_consistency_in_neighborhood_q75\":0.6770833333,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9192105018,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.875,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.90625,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.5,\"target_diversity_in_neighborhood_mean\":0.7987378983,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.7832382117,\"target_diversity_in_neighborhood_q50\":0.9544340029,\"target_diversity_in_neighborhood_q75\":0.9940302115,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.8442272199,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.8112781245,\"approx_diversity_in_neighborhood_q50\":0.9544340029,\"approx_diversity_in_neighborhood_q75\":0.9886994083,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"lin\",\"attack\":\"org\",\"bacc_test\":0.9582931533,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":25.4836363636,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":19.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.513416699,\"uncertainty_q0\":0.064979776,\"uncertainty_q25\":0.3283125105,\"uncertainty_q50\":0.4747779645,\"uncertainty_q75\":0.6709432375,\"uncertainty_q1\":0.9999881312,\"uncertainty_minmax\":0.9350083552,\"target_approx_consistency_in_neighborhood_mean\":0.931769639,\"target_approx_consistency_in_neighborhood_q0\":0.25,\"target_approx_consistency_in_neighborhood_q25\":0.9564393939,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":0.75,\"pred_targets_consistency_in_neighborhood_mean\":0.935998849,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.9354166667,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.9430670804,\"target_targets_consistency_in_neighborhood_q0\":0.375,\"target_targets_consistency_in_neighborhood_q25\":0.9375,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.625,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9609792833,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.9212962963,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.5,\"target_diversity_in_neighborhood_mean\":0.1823666205,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.3372900666,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.189413582,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.2583236403,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"lin\",\"attack\":\"zoo\",\"bacc_test\":0.0964320154,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":23.1090909091,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":15.5,\"neighborhood_size_q50\":26.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.9705028604,\"uncertainty_q0\":0.7482382864,\"uncertainty_q25\":0.9705230169,\"uncertainty_q50\":0.9972887806,\"uncertainty_q75\":0.9997563489,\"uncertainty_q1\":0.9999999999,\"uncertainty_minmax\":0.2517617135,\"target_approx_consistency_in_neighborhood_mean\":0.586307824,\"target_approx_consistency_in_neighborhood_q0\":0.125,\"target_approx_consistency_in_neighborhood_q25\":0.40625,\"target_approx_consistency_in_neighborhood_q50\":0.5769230769,\"target_approx_consistency_in_neighborhood_q75\":0.75,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":0.875,\"pred_targets_consistency_in_neighborhood_mean\":0.4285313439,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.28125,\"pred_targets_consistency_in_neighborhood_q50\":0.4375,\"pred_targets_consistency_in_neighborhood_q75\":0.59375,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.5874359002,\"target_targets_consistency_in_neighborhood_q0\":0.0454545455,\"target_targets_consistency_in_neighborhood_q25\":0.40625,\"target_targets_consistency_in_neighborhood_q50\":0.5625,\"target_targets_consistency_in_neighborhood_q75\":0.75,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.9545454545,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9283143523,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.9032258065,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.9375,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.5,\"target_diversity_in_neighborhood_mean\":0.7903205753,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.7610414845,\"target_diversity_in_neighborhood_q50\":0.9283620724,\"target_diversity_in_neighborhood_q75\":0.985228136,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.7993762529,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.7495952573,\"approx_diversity_in_neighborhood_q50\":0.9283620724,\"approx_diversity_in_neighborhood_q75\":0.9809047718,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"svm\",\"attack\":\"hsj\",\"bacc_test\":0.0,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":26.0145454545,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":21.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.7476172663,\"uncertainty_q0\":0.3752575977,\"uncertainty_q25\":0.6528538924,\"uncertainty_q50\":0.7678617705,\"uncertainty_q75\":0.8514003267,\"uncertainty_q1\":0.9951512433,\"uncertainty_minmax\":0.6198936456,\"target_approx_consistency_in_neighborhood_mean\":0.6519720986,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.5236895161,\"target_approx_consistency_in_neighborhood_q50\":0.7142857143,\"target_approx_consistency_in_neighborhood_q75\":0.84375,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.3480279014,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.15625,\"pred_targets_consistency_in_neighborhood_q50\":0.2857142857,\"pred_targets_consistency_in_neighborhood_q75\":0.4763104839,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.6519720986,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.5236895161,\"target_targets_consistency_in_neighborhood_q50\":0.7142857143,\"target_targets_consistency_in_neighborhood_q75\":0.84375,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q0\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.0,\"target_diversity_in_neighborhood_mean\":0.7100267661,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.5831542382,\"target_diversity_in_neighborhood_q50\":0.8112781245,\"target_diversity_in_neighborhood_q75\":0.9283620724,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.7100267661,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.5831542382,\"approx_diversity_in_neighborhood_q50\":0.8112781245,\"approx_diversity_in_neighborhood_q75\":0.9283620724,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"svm\",\"attack\":\"org\",\"bacc_test\":1.0,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":25.4836363636,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":19.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0204030211,\"uncertainty_q0\":0.0000000053,\"uncertainty_q25\":0.0000488819,\"uncertainty_q50\":0.0004543851,\"uncertainty_q75\":0.0073514649,\"uncertainty_q1\":0.83395812,\"uncertainty_minmax\":0.8339581147,\"target_approx_consistency_in_neighborhood_mean\":0.9430670804,\"target_approx_consistency_in_neighborhood_q0\":0.375,\"target_approx_consistency_in_neighborhood_q25\":0.9375,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":0.625,\"pred_targets_consistency_in_neighborhood_mean\":0.9430670804,\"pred_targets_consistency_in_neighborhood_q0\":0.375,\"pred_targets_consistency_in_neighborhood_q25\":0.9375,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":0.625,\"target_targets_consistency_in_neighborhood_mean\":0.9430670804,\"target_targets_consistency_in_neighborhood_q0\":0.375,\"target_targets_consistency_in_neighborhood_q25\":0.9375,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.625,\"targets_and_approxs_consistency_in_neighborhood_mean\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q0\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.0,\"target_diversity_in_neighborhood_mean\":0.1823666205,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.3372900666,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.1823666205,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.3372900666,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"svm\",\"attack\":\"zoo\",\"bacc_test\":0.1241830065,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":24.6218181818,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":17.0,\"neighborhood_size_q50\":31.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.8605557249,\"uncertainty_q0\":0.0000031358,\"uncertainty_q25\":0.8899982151,\"uncertainty_q50\":0.9983588521,\"uncertainty_q75\":0.9998726587,\"uncertainty_q1\":1.0,\"uncertainty_minmax\":0.9999968642,\"target_approx_consistency_in_neighborhood_mean\":0.696228827,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.5625,\"target_approx_consistency_in_neighborhood_q50\":0.75,\"target_approx_consistency_in_neighborhood_q75\":0.8911764706,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.4230947154,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.1695402299,\"pred_targets_consistency_in_neighborhood_q50\":0.34375,\"pred_targets_consistency_in_neighborhood_q75\":0.6846590909,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.696228827,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.5625,\"target_targets_consistency_in_neighborhood_q50\":0.75,\"target_targets_consistency_in_neighborhood_q75\":0.8911764706,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q0\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.0,\"target_diversity_in_neighborhood_mean\":0.6325261041,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.4488644887,\"target_diversity_in_neighborhood_q50\":0.7871265862,\"target_diversity_in_neighborhood_q75\":0.9233289532,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.6325261041,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.4488644887,\"approx_diversity_in_neighborhood_q50\":0.7871265862,\"approx_diversity_in_neighborhood_q75\":0.9233289532,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"xgb\",\"attack\":\"hsj\",\"bacc_test\":0.0032679739,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":25.2472727273,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":20.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.3790284305,\"uncertainty_q0\":0.0000005629,\"uncertainty_q25\":0.0467629274,\"uncertainty_q50\":0.2654835989,\"uncertainty_q75\":0.709752745,\"uncertainty_q1\":0.9999998203,\"uncertainty_minmax\":0.9999992573,\"target_approx_consistency_in_neighborhood_mean\":0.6194899307,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.4677083333,\"target_approx_consistency_in_neighborhood_q50\":0.6666666667,\"target_approx_consistency_in_neighborhood_q75\":0.8086080586,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.3834587916,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.1913919414,\"pred_targets_consistency_in_neighborhood_q50\":0.3333333333,\"pred_targets_consistency_in_neighborhood_q75\":0.5416666667,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.620177572,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.4666666667,\"target_targets_consistency_in_neighborhood_q50\":0.6666666667,\"target_targets_consistency_in_neighborhood_q75\":0.8110119048,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9977139571,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.8,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.2,\"target_diversity_in_neighborhood_mean\":0.750929798,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.6313249522,\"target_diversity_in_neighborhood_q50\":0.8571484374,\"target_diversity_in_neighborhood_q75\":0.9544340029,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.7519570102,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.6437049604,\"approx_diversity_in_neighborhood_q50\":0.8453509366,\"approx_diversity_in_neighborhood_q75\":0.9544340029,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"xgb\",\"attack\":\"org\",\"bacc_test\":0.9967320261,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":25.4836363636,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":19.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0000463854,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0000000082,\"uncertainty_q50\":0.000000044,\"uncertainty_q75\":0.0000003193,\"uncertainty_q1\":0.00638837,\"uncertainty_minmax\":0.0063883699,\"target_approx_consistency_in_neighborhood_mean\":0.94091832,\"target_approx_consistency_in_neighborhood_q0\":0.375,\"target_approx_consistency_in_neighborhood_q25\":0.9354166667,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":0.625,\"pred_targets_consistency_in_neighborhood_mean\":0.9394307167,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.9375,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.9430670804,\"target_targets_consistency_in_neighborhood_q0\":0.375,\"target_targets_consistency_in_neighborhood_q25\":0.9375,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.625,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9978512397,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.5,\"target_diversity_in_neighborhood_mean\":0.1823666205,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.3372900666,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.187601155,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.3453247008,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"xgb\",\"attack\":\"zoo\",\"bacc_test\":0.1512643309,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":20.5090909091,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":9.0,\"neighborhood_size_q50\":24.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.3131793852,\"uncertainty_q0\":0.0000000003,\"uncertainty_q25\":0.0002505599,\"uncertainty_q50\":0.0856749504,\"uncertainty_q75\":0.6548547007,\"uncertainty_q1\":0.9999993356,\"uncertainty_minmax\":0.9999993353,\"target_approx_consistency_in_neighborhood_mean\":0.6420298219,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.4,\"target_approx_consistency_in_neighborhood_q50\":0.6785714286,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.5162740949,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.25,\"pred_targets_consistency_in_neighborhood_q50\":0.5,\"pred_targets_consistency_in_neighborhood_q75\":0.7871767241,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.6437259051,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.4,\"target_targets_consistency_in_neighborhood_q50\":0.6785714286,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9969518669,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.5,\"target_diversity_in_neighborhood_mean\":0.5657871996,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.7793498373,\"target_diversity_in_neighborhood_q75\":0.9283620724,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.5676521579,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.7706290694,\"approx_diversity_in_neighborhood_q75\":0.9283620724,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"lin\",\"attack\":\"hsj\",\"bacc_test\":0.3242592593,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":23.7922077922,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":15.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.9931065222,\"uncertainty_q0\":0.9308419516,\"uncertainty_q25\":0.9902877593,\"uncertainty_q50\":0.9960606929,\"uncertainty_q75\":0.999014028,\"uncertainty_q1\":0.9999999377,\"uncertainty_minmax\":0.0691579861,\"target_approx_consistency_in_neighborhood_mean\":0.5853737447,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.3844827586,\"target_approx_consistency_in_neighborhood_q50\":0.6583333333,\"target_approx_consistency_in_neighborhood_q75\":0.78125,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.4527318437,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.34375,\"pred_targets_consistency_in_neighborhood_q50\":0.4375,\"pred_targets_consistency_in_neighborhood_q75\":0.5607638889,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.5545189847,\"target_targets_consistency_in_neighborhood_q0\":0.2,\"target_targets_consistency_in_neighborhood_q25\":0.4411111111,\"target_targets_consistency_in_neighborhood_q50\":0.5625,\"target_targets_consistency_in_neighborhood_q75\":0.65625,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.8,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.6863886308,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.625,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.6875,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.75,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.9190442878,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.8960382325,\"target_diversity_in_neighborhood_q50\":0.9544340029,\"target_diversity_in_neighborhood_q75\":0.9886994083,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.7932811562,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.6962122601,\"approx_diversity_in_neighborhood_q50\":0.8571484374,\"approx_diversity_in_neighborhood_q75\":0.9544340029,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"lin\",\"attack\":\"org\",\"bacc_test\":0.6757407407,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":23.5194805195,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":14.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.77581969,\"uncertainty_q0\":0.0883268472,\"uncertainty_q25\":0.6377381403,\"uncertainty_q50\":0.8272850508,\"uncertainty_q75\":0.9482398223,\"uncertainty_q1\":0.9999892001,\"uncertainty_minmax\":0.9116623529,\"target_approx_consistency_in_neighborhood_mean\":0.6865661853,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.4545454545,\"target_approx_consistency_in_neighborhood_q50\":0.8181818182,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.6961026081,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.5625,\"pred_targets_consistency_in_neighborhood_q50\":0.71875,\"pred_targets_consistency_in_neighborhood_q75\":0.875,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.6651322257,\"target_targets_consistency_in_neighborhood_q0\":0.09375,\"target_targets_consistency_in_neighborhood_q25\":0.5046296296,\"target_targets_consistency_in_neighborhood_q50\":0.7165178571,\"target_targets_consistency_in_neighborhood_q75\":0.875,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.90625,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.750956215,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.643907563,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.75,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.875,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.7411894884,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.5435644432,\"target_diversity_in_neighborhood_q50\":0.8283145305,\"target_diversity_in_neighborhood_q75\":0.969232637,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.4348051071,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.3372900666,\"approx_diversity_in_neighborhood_q75\":0.8786842917,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"lin\",\"attack\":\"zoo\",\"bacc_test\":0.0648148148,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":21.1168831169,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":11.0,\"neighborhood_size_q50\":22.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.9651628771,\"uncertainty_q0\":0.5219496677,\"uncertainty_q25\":0.969854699,\"uncertainty_q50\":0.9929758436,\"uncertainty_q75\":0.9991594095,\"uncertainty_q1\":0.9999985912,\"uncertainty_minmax\":0.4780489235,\"target_approx_consistency_in_neighborhood_mean\":0.538969653,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.3460477941,\"target_approx_consistency_in_neighborhood_q50\":0.6,\"target_approx_consistency_in_neighborhood_q75\":0.75,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.4906294477,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.3684210526,\"pred_targets_consistency_in_neighborhood_q50\":0.4772256729,\"pred_targets_consistency_in_neighborhood_q75\":0.59375,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.5198623631,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.4090909091,\"target_targets_consistency_in_neighborhood_q50\":0.5370879121,\"target_targets_consistency_in_neighborhood_q75\":0.6315789474,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.6730840756,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.6092995169,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.6875,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.75,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.883277396,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.8658356885,\"target_diversity_in_neighborhood_q50\":0.9544340029,\"target_diversity_in_neighborhood_q75\":0.9886994083,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.714365586,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.6252624052,\"approx_diversity_in_neighborhood_q50\":0.8409958393,\"approx_diversity_in_neighborhood_q75\":0.9449160258,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"svm\",\"attack\":\"hsj\",\"bacc_test\":0.2987037037,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":24.2792207792,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":16.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.984060104,\"uncertainty_q0\":0.9041817325,\"uncertainty_q25\":0.9786913044,\"uncertainty_q50\":0.9920200069,\"uncertainty_q75\":0.9979471855,\"uncertainty_q1\":1.0,\"uncertainty_minmax\":0.0958182675,\"target_approx_consistency_in_neighborhood_mean\":0.5818839124,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.40625,\"target_approx_consistency_in_neighborhood_q50\":0.6220238095,\"target_approx_consistency_in_neighborhood_q75\":0.78125,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.4175208785,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.28125,\"pred_targets_consistency_in_neighborhood_q50\":0.40625,\"pred_targets_consistency_in_neighborhood_q75\":0.53125,\"pred_targets_consistency_in_neighborhood_q1\":0.8,\"pred_targets_consistency_in_neighborhood_minmax\":0.8,\"target_targets_consistency_in_neighborhood_mean\":0.5818185135,\"target_targets_consistency_in_neighborhood_q0\":0.21875,\"target_targets_consistency_in_neighborhood_q25\":0.46875,\"target_targets_consistency_in_neighborhood_q50\":0.5909926471,\"target_targets_consistency_in_neighborhood_q75\":0.71875,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.78125,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.6855580674,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.59375,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.6875,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.75,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.8819589403,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.8571484374,\"target_diversity_in_neighborhood_q50\":0.9233289532,\"target_diversity_in_neighborhood_q75\":0.9886994083,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.7906724622,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.6962122601,\"approx_diversity_in_neighborhood_q50\":0.8571484374,\"approx_diversity_in_neighborhood_q75\":0.9633682248,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"svm\",\"attack\":\"org\",\"bacc_test\":0.7012962963,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":23.5194805195,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":14.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.6979254811,\"uncertainty_q0\":0.2078876074,\"uncertainty_q25\":0.4931940474,\"uncertainty_q50\":0.7420771248,\"uncertainty_q75\":0.9005873367,\"uncertainty_q1\":1.0,\"uncertainty_minmax\":0.7921123926,\"target_approx_consistency_in_neighborhood_mean\":0.6914812428,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.4562937063,\"target_approx_consistency_in_neighborhood_q50\":0.78125,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.6953800756,\"pred_targets_consistency_in_neighborhood_q0\":0.2142857143,\"pred_targets_consistency_in_neighborhood_q25\":0.5555555556,\"pred_targets_consistency_in_neighborhood_q50\":0.71875,\"pred_targets_consistency_in_neighborhood_q75\":0.875,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":0.7857142857,\"target_targets_consistency_in_neighborhood_mean\":0.6651322257,\"target_targets_consistency_in_neighborhood_q0\":0.09375,\"target_targets_consistency_in_neighborhood_q25\":0.5046296296,\"target_targets_consistency_in_neighborhood_q50\":0.7165178571,\"target_targets_consistency_in_neighborhood_q75\":0.875,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.90625,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.7558781805,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.25,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.6428571429,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.75,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.875,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.75,\"target_diversity_in_neighborhood_mean\":0.7411894884,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.5435644432,\"target_diversity_in_neighborhood_q50\":0.8283145305,\"target_diversity_in_neighborhood_q75\":0.969232637,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.4868528814,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.5435644432,\"approx_diversity_in_neighborhood_q75\":0.8739810481,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"svm\",\"attack\":\"zoo\",\"bacc_test\":0.1562962963,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":21.6883116883,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":11.0,\"neighborhood_size_q50\":26.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.9281896205,\"uncertainty_q0\":0.3062091413,\"uncertainty_q25\":0.9115964886,\"uncertainty_q50\":0.9802783805,\"uncertainty_q75\":0.9970049699,\"uncertainty_q1\":1.0,\"uncertainty_minmax\":0.6937908587,\"target_approx_consistency_in_neighborhood_mean\":0.5811516165,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.34375,\"target_approx_consistency_in_neighborhood_q50\":0.6282894737,\"target_approx_consistency_in_neighborhood_q75\":0.84375,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.5071486194,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.3333333333,\"pred_targets_consistency_in_neighborhood_q50\":0.5,\"pred_targets_consistency_in_neighborhood_q75\":0.65625,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.5786091181,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.4375,\"target_targets_consistency_in_neighborhood_q50\":0.5916666667,\"target_targets_consistency_in_neighborhood_q75\":0.7458333333,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.7067111304,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.25,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.6092032967,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.6875,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.7953125,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.75,\"target_diversity_in_neighborhood_mean\":0.8300929505,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.8112781245,\"target_diversity_in_neighborhood_q50\":0.9283620724,\"target_diversity_in_neighborhood_q75\":0.9838882912,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.6724206925,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.5435644432,\"approx_diversity_in_neighborhood_q50\":0.7578784625,\"approx_diversity_in_neighborhood_q75\":0.9375553755,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"xgb\",\"attack\":\"hsj\",\"bacc_test\":0.2894444444,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":22.538961039,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":11.25,\"neighborhood_size_q50\":30.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.8483512345,\"uncertainty_q0\":0.1248662949,\"uncertainty_q25\":0.7888778647,\"uncertainty_q50\":0.9449394853,\"uncertainty_q75\":0.9930294847,\"uncertainty_q1\":0.9999989529,\"uncertainty_minmax\":0.875132658,\"target_approx_consistency_in_neighborhood_mean\":0.6480567545,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.4861111111,\"target_approx_consistency_in_neighborhood_q50\":0.6858552632,\"target_approx_consistency_in_neighborhood_q75\":0.8854166667,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.3532916119,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.1875,\"pred_targets_consistency_in_neighborhood_q50\":0.3125,\"pred_targets_consistency_in_neighborhood_q75\":0.5,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.6366294141,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.5,\"target_targets_consistency_in_neighborhood_q50\":0.6666666667,\"target_targets_consistency_in_neighborhood_q75\":0.8125,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.7419332302,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.6428571429,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.75,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.84375,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.7723754962,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.6962122601,\"target_diversity_in_neighborhood_q50\":0.8571484374,\"target_diversity_in_neighborhood_q75\":0.9744894034,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.6679257869,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.4488644887,\"approx_diversity_in_neighborhood_q50\":0.764253766,\"approx_diversity_in_neighborhood_q75\":0.9659836656,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"xgb\",\"attack\":\"org\",\"bacc_test\":0.7105555556,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":23.5194805195,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":14.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.3235317273,\"uncertainty_q0\":0.0012200234,\"uncertainty_q25\":0.0366432414,\"uncertainty_q50\":0.1907841319,\"uncertainty_q75\":0.5392620463,\"uncertainty_q1\":0.9997239945,\"uncertainty_minmax\":0.9985039711,\"target_approx_consistency_in_neighborhood_mean\":0.6776543098,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.5,\"target_approx_consistency_in_neighborhood_q50\":0.7165178571,\"target_approx_consistency_in_neighborhood_q75\":0.9661458333,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.6717582536,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.5,\"pred_targets_consistency_in_neighborhood_q50\":0.7165178571,\"pred_targets_consistency_in_neighborhood_q75\":0.875,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.6651322257,\"target_targets_consistency_in_neighborhood_q0\":0.09375,\"target_targets_consistency_in_neighborhood_q25\":0.5046296296,\"target_targets_consistency_in_neighborhood_q50\":0.7165178571,\"target_targets_consistency_in_neighborhood_q75\":0.875,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.90625,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.7493388222,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.6428571429,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.7795138889,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.875,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.7411894884,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.5435644432,\"target_diversity_in_neighborhood_q50\":0.8283145305,\"target_diversity_in_neighborhood_q75\":0.969232637,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.5945551392,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.2006223243,\"approx_diversity_in_neighborhood_q50\":0.7090701775,\"approx_diversity_in_neighborhood_q75\":0.9544340029,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"xgb\",\"attack\":\"zoo\",\"bacc_test\":0.1035185185,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":19.7662337662,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":9.0,\"neighborhood_size_q50\":22.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.6393430452,\"uncertainty_q0\":0.0011893617,\"uncertainty_q25\":0.3200501763,\"uncertainty_q50\":0.7636684045,\"uncertainty_q75\":0.961560774,\"uncertainty_q1\":0.9997919264,\"uncertainty_minmax\":0.9986025648,\"target_approx_consistency_in_neighborhood_mean\":0.6180773919,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.5,\"target_approx_consistency_in_neighborhood_q50\":0.6055555556,\"target_approx_consistency_in_neighborhood_q75\":0.8609375,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.4366249274,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.25,\"pred_targets_consistency_in_neighborhood_q50\":0.421182266,\"pred_targets_consistency_in_neighborhood_q75\":0.6171875,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.6324991754,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.5,\"target_targets_consistency_in_neighborhood_q50\":0.6363636364,\"target_targets_consistency_in_neighborhood_q75\":0.78125,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.7312556865,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.625,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.71875,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.84375,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.7421285891,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.6962122601,\"target_diversity_in_neighborhood_q50\":0.8904916402,\"target_diversity_in_neighborhood_q75\":0.9749616496,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.6741990258,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.4488644887,\"approx_diversity_in_neighborhood_q50\":0.8112781245,\"approx_diversity_in_neighborhood_q75\":0.985228136,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"wilt\",\"model\":\"lin\",\"attack\":\"hsj\",\"bacc_test\":0.5,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":22.7799586777,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":12.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.295060918,\"uncertainty_q0\":0.066663795,\"uncertainty_q25\":0.2344353998,\"uncertainty_q50\":0.2835336722,\"uncertainty_q75\":0.3514480051,\"uncertainty_q1\":0.5899475505,\"uncertainty_minmax\":0.5232837555,\"target_approx_consistency_in_neighborhood_mean\":0.9431818182,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":1.0,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.9390907989,\"pred_targets_consistency_in_neighborhood_q0\":0.5,\"pred_targets_consistency_in_neighborhood_q25\":0.90625,\"pred_targets_consistency_in_neighborhood_q50\":0.96875,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":0.5,\"target_targets_consistency_in_neighborhood_mean\":0.902328034,\"target_targets_consistency_in_neighborhood_q0\":0.03125,\"target_targets_consistency_in_neighborhood_q25\":0.90625,\"target_targets_consistency_in_neighborhood_q50\":0.96875,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9390907989,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.90625,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.5,\"target_diversity_in_neighborhood_mean\":0.2534039636,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.2006223243,\"target_diversity_in_neighborhood_q75\":0.4488644887,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.0,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.0,\"approx_diversity_in_neighborhood_minmax\":0.0},{\"dataset\":\"wilt\",\"model\":\"lin\",\"attack\":\"org\",\"bacc_test\":0.5,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":22.7799586777,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":12.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.2946472526,\"uncertainty_q0\":0.0645033088,\"uncertainty_q25\":0.2338169922,\"uncertainty_q50\":0.2830698838,\"uncertainty_q75\":0.3508541172,\"uncertainty_q1\":0.5900787609,\"uncertainty_minmax\":0.5255754521,\"target_approx_consistency_in_neighborhood_mean\":0.9431818182,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":1.0,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.9390907989,\"pred_targets_consistency_in_neighborhood_q0\":0.5,\"pred_targets_consistency_in_neighborhood_q25\":0.90625,\"pred_targets_consistency_in_neighborhood_q50\":0.96875,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":0.5,\"target_targets_consistency_in_neighborhood_mean\":0.902328034,\"target_targets_consistency_in_neighborhood_q0\":0.03125,\"target_targets_consistency_in_neighborhood_q25\":0.90625,\"target_targets_consistency_in_neighborhood_q50\":0.96875,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9390907989,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.90625,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.5,\"target_diversity_in_neighborhood_mean\":0.2534039636,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.2006223243,\"target_diversity_in_neighborhood_q75\":0.4488644887,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.0,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.0,\"approx_diversity_in_neighborhood_minmax\":0.0},{\"dataset\":\"wilt\",\"model\":\"lin\",\"attack\":\"zoo\",\"bacc_test\":0.5,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":2.5743801653,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":1.0,\"neighborhood_size_q75\":1.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.7969932054,\"uncertainty_q0\":0.227339489,\"uncertainty_q25\":0.7803725852,\"uncertainty_q50\":0.8373089912,\"uncertainty_q75\":0.8478445983,\"uncertainty_q1\":0.9045166748,\"uncertainty_minmax\":0.6771771857,\"target_approx_consistency_in_neighborhood_mean\":0.9431818182,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":1.0,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.9237274568,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":1.0,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.8872229564,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":1.0,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9237274568,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.0361107347,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.0,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.0,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.0,\"approx_diversity_in_neighborhood_minmax\":0.0},{\"dataset\":\"wilt\",\"model\":\"svm\",\"attack\":\"hsj\",\"bacc_test\":0.1351588171,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":22.5599173554,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":12.0,\"neighborhood_size_q50\":31.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.6184209319,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.4446313573,\"uncertainty_q50\":0.7500934781,\"uncertainty_q75\":0.8697512423,\"uncertainty_q1\":1.0,\"uncertainty_minmax\":1.0,\"target_approx_consistency_in_neighborhood_mean\":0.8878914242,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.875,\"target_approx_consistency_in_neighborhood_q50\":0.9545454545,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.2805544298,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.0,\"pred_targets_consistency_in_neighborhood_q50\":0.0625,\"pred_targets_consistency_in_neighborhood_q75\":0.6953125,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.8980434769,\"target_targets_consistency_in_neighborhood_q0\":0.03125,\"target_targets_consistency_in_neighborhood_q25\":0.90625,\"target_targets_consistency_in_neighborhood_q50\":0.96875,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9795884228,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.2658997013,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.2006223243,\"target_diversity_in_neighborhood_q75\":0.4488644887,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.297690108,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.2539504807,\"approx_diversity_in_neighborhood_q75\":0.5032583348,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"wilt\",\"model\":\"svm\",\"attack\":\"org\",\"bacc_test\":0.9559693319,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":22.7799586777,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":12.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0685009461,\"uncertainty_q0\":0.0000000118,\"uncertainty_q25\":0.0002737436,\"uncertainty_q50\":0.0044162421,\"uncertainty_q75\":0.0327941618,\"uncertainty_q1\":0.9996173998,\"uncertainty_minmax\":0.999617388,\"target_approx_consistency_in_neighborhood_mean\":0.8931661235,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.8888888889,\"target_approx_consistency_in_neighborhood_q50\":0.9672043011,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.8933344395,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.90625,\"pred_targets_consistency_in_neighborhood_q50\":0.96875,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.902328034,\"target_targets_consistency_in_neighborhood_q0\":0.03125,\"target_targets_consistency_in_neighborhood_q25\":0.90625,\"target_targets_consistency_in_neighborhood_q50\":0.96875,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9807507091,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.2534039636,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.2006223243,\"target_diversity_in_neighborhood_q75\":0.4488644887,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.2833595701,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.2055925082,\"approx_diversity_in_neighborhood_q75\":0.4586858162,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"wilt\",\"model\":\"svm\",\"attack\":\"zoo\",\"bacc_test\":0.0164293538,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":8.5702479339,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":2.0,\"neighborhood_size_q50\":7.0,\"neighborhood_size_q75\":13.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0659017344,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0,\"uncertainty_q50\":0.0,\"uncertainty_q75\":0.0,\"uncertainty_q1\":0.9998981519,\"uncertainty_minmax\":0.9998981519,\"target_approx_consistency_in_neighborhood_mean\":0.7935725553,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.7142857143,\"target_approx_consistency_in_neighborhood_q50\":0.8918128655,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.211654094,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.0,\"pred_targets_consistency_in_neighborhood_q50\":0.1,\"pred_targets_consistency_in_neighborhood_q75\":0.2955882353,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.8193376415,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.75,\"target_targets_consistency_in_neighborhood_q50\":0.9272486772,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9627654922,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.3510866121,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.7219280949,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.3753255838,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.3622958308,\"approx_diversity_in_neighborhood_q75\":0.7642045065,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"wilt\",\"model\":\"xgb\",\"attack\":\"hsj\",\"bacc_test\":0.1112814896,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":22.8429752066,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":12.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.6082987319,\"uncertainty_q0\":0.0000000002,\"uncertainty_q25\":0.3368488342,\"uncertainty_q50\":0.6579846484,\"uncertainty_q75\":0.921324727,\"uncertainty_q1\":0.9999999322,\"uncertainty_minmax\":0.999999932,\"target_approx_consistency_in_neighborhood_mean\":0.9010924543,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.90625,\"target_approx_consistency_in_neighborhood_q50\":0.96875,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.0990455012,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.0,\"pred_targets_consistency_in_neighborhood_q50\":0.03125,\"pred_targets_consistency_in_neighborhood_q75\":0.1039019964,\"pred_targets_consistency_in_neighborhood_q1\":0.96875,\"pred_targets_consistency_in_neighborhood_minmax\":0.96875,\"target_targets_consistency_in_neighborhood_mean\":0.8947281651,\"target_targets_consistency_in_neighborhood_q0\":0.03125,\"target_targets_consistency_in_neighborhood_q25\":0.887254902,\"target_targets_consistency_in_neighborhood_q50\":0.96875,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9812584348,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.7272727273,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.2727272727,\"target_diversity_in_neighborhood_mean\":0.2784626137,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.2006223243,\"target_diversity_in_neighborhood_q75\":0.4537163392,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.2512693897,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.2006223243,\"approx_diversity_in_neighborhood_q75\":0.4488644887,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"wilt\",\"model\":\"xgb\",\"attack\":\"org\",\"bacc_test\":0.8887185104,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":22.7799586777,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":12.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0214004835,\"uncertainty_q0\":0.0000013903,\"uncertainty_q25\":0.0005207739,\"uncertainty_q50\":0.0011419314,\"uncertainty_q75\":0.0027503582,\"uncertainty_q1\":0.9911242512,\"uncertainty_minmax\":0.9911228609,\"target_approx_consistency_in_neighborhood_mean\":0.9083173183,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.9083806818,\"target_approx_consistency_in_neighborhood_q50\":0.96875,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.9086133067,\"pred_targets_consistency_in_neighborhood_q0\":0.03125,\"pred_targets_consistency_in_neighborhood_q25\":0.90625,\"pred_targets_consistency_in_neighborhood_q50\":0.96875,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":0.96875,\"target_targets_consistency_in_neighborhood_mean\":0.902328034,\"target_targets_consistency_in_neighborhood_q0\":0.03125,\"target_targets_consistency_in_neighborhood_q25\":0.90625,\"target_targets_consistency_in_neighborhood_q50\":0.96875,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9822712423,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.6666666667,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.3333333333,\"target_diversity_in_neighborhood_mean\":0.2534039636,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.2006223243,\"target_diversity_in_neighborhood_q75\":0.4488644887,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.2275596192,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.2006223243,\"approx_diversity_in_neighborhood_q75\":0.3736608914,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"wilt\",\"model\":\"xgb\",\"attack\":\"zoo\",\"bacc_test\":0.0394304491,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":9.8904958678,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":2.0,\"neighborhood_size_q50\":7.0,\"neighborhood_size_q75\":15.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0466145885,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0000000143,\"uncertainty_q50\":0.0000040901,\"uncertainty_q75\":0.0013760283,\"uncertainty_q1\":0.9996229404,\"uncertainty_minmax\":0.9996229404,\"target_approx_consistency_in_neighborhood_mean\":0.845496402,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.8,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.2383607615,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.0,\"pred_targets_consistency_in_neighborhood_q50\":0.0714285714,\"pred_targets_consistency_in_neighborhood_q75\":0.3333333333,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.8336102427,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.7777777778,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9724676601,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.2992071792,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.6500224216,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.2677711974,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.5435644432,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"nn\",\"attack\":\"bim\",\"bacc_test\":0.0,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":8.7963636364,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":5.0,\"neighborhood_size_q75\":14.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0017398332,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0,\"uncertainty_q50\":0.0,\"uncertainty_q75\":0.0000000004,\"uncertainty_q1\":0.3808226311,\"uncertainty_minmax\":0.3808226311,\"target_approx_consistency_in_neighborhood_mean\":0.0487875864,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.0,\"target_approx_consistency_in_neighborhood_q50\":0.0,\"target_approx_consistency_in_neighborhood_q75\":0.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.9512124136,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":1.0,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.0487875864,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.0,\"target_targets_consistency_in_neighborhood_q50\":0.0,\"target_targets_consistency_in_neighborhood_q75\":0.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q0\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.0,\"target_diversity_in_neighborhood_mean\":0.0630930284,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.0,\"target_diversity_in_neighborhood_q1\":0.9983636726,\"target_diversity_in_neighborhood_minmax\":0.9983636726,\"approx_diversity_in_neighborhood_mean\":0.0630930284,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.9983636726,\"approx_diversity_in_neighborhood_minmax\":0.9983636726},{\"dataset\":\"banknote\",\"model\":\"nn\",\"attack\":\"fgm\",\"bacc_test\":0.0,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":7.5309090909,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":5.0,\"neighborhood_size_q75\":12.0,\"neighborhood_size_q1\":30.0,\"neighborhood_size_minmax\":29.0,\"uncertainty_mean\":0.001288313,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0,\"uncertainty_q50\":0.0,\"uncertainty_q75\":0.0000000005,\"uncertainty_q1\":0.2478158209,\"uncertainty_minmax\":0.2478158208,\"target_approx_consistency_in_neighborhood_mean\":0.0487875864,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.0,\"target_approx_consistency_in_neighborhood_q50\":0.0,\"target_approx_consistency_in_neighborhood_q75\":0.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.9512124136,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":1.0,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.0487875864,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.0,\"target_targets_consistency_in_neighborhood_q50\":0.0,\"target_targets_consistency_in_neighborhood_q75\":0.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q0\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.0,\"target_diversity_in_neighborhood_mean\":0.0630930284,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.0,\"target_diversity_in_neighborhood_q1\":0.9983636726,\"target_diversity_in_neighborhood_minmax\":0.9983636726,\"approx_diversity_in_neighborhood_mean\":0.0630930284,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.9983636726,\"approx_diversity_in_neighborhood_minmax\":0.9983636726},{\"dataset\":\"banknote\",\"model\":\"nn\",\"attack\":\"org\",\"bacc_test\":1.0,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":25.4872727273,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":19.0,\"neighborhood_size_q50\":32.0,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0078923911,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0000000234,\"uncertainty_q50\":0.0000004075,\"uncertainty_q75\":0.0000428935,\"uncertainty_q1\":0.5204205763,\"uncertainty_minmax\":0.5204205763,\"target_approx_consistency_in_neighborhood_mean\":0.9433144066,\"target_approx_consistency_in_neighborhood_q0\":0.375,\"target_approx_consistency_in_neighborhood_q25\":0.9375,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":0.625,\"pred_targets_consistency_in_neighborhood_mean\":0.9433144066,\"pred_targets_consistency_in_neighborhood_q0\":0.375,\"pred_targets_consistency_in_neighborhood_q25\":0.9375,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":0.625,\"target_targets_consistency_in_neighborhood_mean\":0.9433144066,\"target_targets_consistency_in_neighborhood_q0\":0.375,\"target_targets_consistency_in_neighborhood_q25\":0.9375,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.625,\"targets_and_approxs_consistency_in_neighborhood_mean\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q0\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.0,\"target_diversity_in_neighborhood_mean\":0.1822932961,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.3372900666,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.1822932961,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.3372900666,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"banknote\",\"model\":\"nn\",\"attack\":\"pgd\",\"bacc_test\":0.0,\"n_test\":275,\"n_classes\":2,\"neighborhood_size_mean\":7.5309090909,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":5.0,\"neighborhood_size_q75\":12.0,\"neighborhood_size_q1\":30.0,\"neighborhood_size_minmax\":29.0,\"uncertainty_mean\":0.000513264,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0,\"uncertainty_q50\":0.0,\"uncertainty_q75\":0.0000000001,\"uncertainty_q1\":0.090579867,\"uncertainty_minmax\":0.090579867,\"target_approx_consistency_in_neighborhood_mean\":0.0487875864,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.0,\"target_approx_consistency_in_neighborhood_q50\":0.0,\"target_approx_consistency_in_neighborhood_q75\":0.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.9512124136,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":1.0,\"pred_targets_consistency_in_neighborhood_q50\":1.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.0487875864,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.0,\"target_targets_consistency_in_neighborhood_q50\":0.0,\"target_targets_consistency_in_neighborhood_q75\":0.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q0\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.0,\"target_diversity_in_neighborhood_mean\":0.0630930284,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.0,\"target_diversity_in_neighborhood_q1\":0.9983636726,\"target_diversity_in_neighborhood_minmax\":0.9983636726,\"approx_diversity_in_neighborhood_mean\":0.0630930284,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.9983636726,\"approx_diversity_in_neighborhood_minmax\":0.9983636726},{\"dataset\":\"diabetes\",\"model\":\"nn\",\"attack\":\"bim\",\"bacc_test\":0.2781481481,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":9.961038961,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":3.0,\"neighborhood_size_q50\":7.0,\"neighborhood_size_q75\":16.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.3439997633,\"uncertainty_q0\":0.0024006044,\"uncertainty_q25\":0.0758199078,\"uncertainty_q50\":0.3305024809,\"uncertainty_q75\":0.5447010547,\"uncertainty_q1\":0.9999946593,\"uncertainty_minmax\":0.9975940549,\"target_approx_consistency_in_neighborhood_mean\":0.525229962,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.2055555556,\"target_approx_consistency_in_neighborhood_q50\":0.5590277778,\"target_approx_consistency_in_neighborhood_q75\":0.825,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.6383980555,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.5,\"pred_targets_consistency_in_neighborhood_q50\":0.6363636364,\"pred_targets_consistency_in_neighborhood_q75\":0.8671875,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.525547548,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.3333333333,\"target_targets_consistency_in_neighborhood_q50\":0.5,\"target_targets_consistency_in_neighborhood_q75\":0.7425,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.7084017317,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.5714285714,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.7142857143,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.6304140219,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.860134503,\"target_diversity_in_neighborhood_q75\":0.985228136,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.5256842228,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.7219280949,\"approx_diversity_in_neighborhood_q75\":0.9544340029,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"nn\",\"attack\":\"fgm\",\"bacc_test\":0.3201851852,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":8.1103896104,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":5.0,\"neighborhood_size_q75\":12.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.3870757708,\"uncertainty_q0\":0.0005187217,\"uncertainty_q25\":0.1336136797,\"uncertainty_q50\":0.3287497958,\"uncertainty_q75\":0.5932731745,\"uncertainty_q1\":0.9989917764,\"uncertainty_minmax\":0.9984730547,\"target_approx_consistency_in_neighborhood_mean\":0.5226266669,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.0,\"target_approx_consistency_in_neighborhood_q50\":0.5486111111,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.6843626418,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.5,\"pred_targets_consistency_in_neighborhood_q50\":0.7142857143,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.4720049976,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.2,\"target_targets_consistency_in_neighborhood_q50\":0.4522727273,\"target_targets_consistency_in_neighborhood_q75\":0.7395833333,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.684907939,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.5,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.7,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.5353527892,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.7399032787,\"target_diversity_in_neighborhood_q75\":0.9595359543,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.4010418451,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.8794634365,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"nn\",\"attack\":\"org\",\"bacc_test\":0.6757407407,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":23.5194805195,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":14.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.5201870236,\"uncertainty_q0\":0.0002297424,\"uncertainty_q25\":0.1296046707,\"uncertainty_q50\":0.5653219079,\"uncertainty_q75\":0.8963621046,\"uncertainty_q1\":0.9996733129,\"uncertainty_minmax\":0.9994435704,\"target_approx_consistency_in_neighborhood_mean\":0.693559132,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.4511363636,\"target_approx_consistency_in_neighborhood_q50\":0.80625,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.7023939909,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.5647321429,\"pred_targets_consistency_in_neighborhood_q50\":0.71875,\"pred_targets_consistency_in_neighborhood_q75\":0.875,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.6651322257,\"target_targets_consistency_in_neighborhood_q0\":0.09375,\"target_targets_consistency_in_neighborhood_q25\":0.5046296296,\"target_targets_consistency_in_neighborhood_q50\":0.7165178571,\"target_targets_consistency_in_neighborhood_q75\":0.875,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.90625,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.7564016365,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.65625,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.7619047619,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.875,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.7411894884,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.5435644432,\"target_diversity_in_neighborhood_q50\":0.8283145305,\"target_diversity_in_neighborhood_q75\":0.969232637,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.4731877064,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.5330619089,\"approx_diversity_in_neighborhood_q75\":0.8571484374,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"diabetes\",\"model\":\"nn\",\"attack\":\"pgd\",\"bacc_test\":0.3372222222,\"n_test\":154,\"n_classes\":2,\"neighborhood_size_mean\":11.6298701299,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":4.0,\"neighborhood_size_q50\":8.0,\"neighborhood_size_q75\":17.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.3099581967,\"uncertainty_q0\":0.0086791873,\"uncertainty_q25\":0.0819315071,\"uncertainty_q50\":0.2286200003,\"uncertainty_q75\":0.4589580385,\"uncertainty_q1\":0.998792803,\"uncertainty_minmax\":0.9901136157,\"target_approx_consistency_in_neighborhood_mean\":0.5477883435,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.25,\"target_approx_consistency_in_neighborhood_q50\":0.6,\"target_approx_consistency_in_neighborhood_q75\":0.8574892241,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.6083616926,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.4015625,\"pred_targets_consistency_in_neighborhood_q50\":0.6125,\"pred_targets_consistency_in_neighborhood_q75\":0.8,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.5665563038,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.3664772727,\"target_targets_consistency_in_neighborhood_q50\":0.5741758242,\"target_targets_consistency_in_neighborhood_q75\":0.75,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.6842491775,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.5339285714,\"targets_and_approxs_consistency_in_neighborhood_q50\":0.7,\"targets_and_approxs_consistency_in_neighborhood_q75\":0.9068181818,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.6868904771,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.5032583348,\"target_diversity_in_neighborhood_q50\":0.883915896,\"target_diversity_in_neighborhood_q75\":0.9709505945,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.5245946342,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.7090701775,\"approx_diversity_in_neighborhood_q75\":0.9182958341,\"approx_diversity_in_neighborhood_q1\":1.0,\"approx_diversity_in_neighborhood_minmax\":1.0},{\"dataset\":\"wilt\",\"model\":\"nn\",\"attack\":\"bim\",\"bacc_test\":0.0856516977,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":1.0,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":1.0,\"neighborhood_size_q75\":1.0,\"neighborhood_size_q1\":1.0,\"neighborhood_size_minmax\":0.0,\"uncertainty_mean\":0.0186263954,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0196563909,\"uncertainty_q50\":0.0196563909,\"uncertainty_q75\":0.0196563909,\"uncertainty_q1\":0.0644094172,\"uncertainty_minmax\":0.0644094172,\"target_approx_consistency_in_neighborhood_mean\":0.7407024793,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":0.0,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.2665289256,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.0,\"pred_targets_consistency_in_neighborhood_q50\":0.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.7355371901,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":0.0,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.992768595,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.0,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.0,\"target_diversity_in_neighborhood_q1\":0.0,\"target_diversity_in_neighborhood_minmax\":0.0,\"approx_diversity_in_neighborhood_mean\":0.0,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.0,\"approx_diversity_in_neighborhood_minmax\":0.0},{\"dataset\":\"wilt\",\"model\":\"nn\",\"attack\":\"fgm\",\"bacc_test\":0.0649507119,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":1.0320247934,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":1.0,\"neighborhood_size_q75\":1.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0182092459,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0127502205,\"uncertainty_q50\":0.0127502205,\"uncertainty_q75\":0.0127502205,\"uncertainty_q1\":0.8756378014,\"uncertainty_minmax\":0.8756378014,\"target_approx_consistency_in_neighborhood_mean\":0.9422456095,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":1.0,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.0941051136,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.0,\"pred_targets_consistency_in_neighborhood_q50\":0.0,\"pred_targets_consistency_in_neighborhood_q75\":0.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.9215844525,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":1.0,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9772727273,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.000463703,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.0,\"target_diversity_in_neighborhood_q1\":0.4488644887,\"target_diversity_in_neighborhood_minmax\":0.4488644887,\"approx_diversity_in_neighborhood_mean\":0.000463703,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.4488644887,\"approx_diversity_in_neighborhood_minmax\":0.4488644887},{\"dataset\":\"wilt\",\"model\":\"nn\",\"attack\":\"org\",\"bacc_test\":0.9069003286,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":22.7799586777,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":12.0,\"neighborhood_size_q50\":31.5,\"neighborhood_size_q75\":32.0,\"neighborhood_size_q1\":32.0,\"neighborhood_size_minmax\":31.0,\"uncertainty_mean\":0.0378024823,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0000493637,\"uncertainty_q50\":0.0007374263,\"uncertainty_q75\":0.0057754285,\"uncertainty_q1\":0.9999433952,\"uncertainty_minmax\":0.9999433952,\"target_approx_consistency_in_neighborhood_mean\":0.9062157145,\"target_approx_consistency_in_neighborhood_q0\":0.03125,\"target_approx_consistency_in_neighborhood_q25\":0.90625,\"target_approx_consistency_in_neighborhood_q50\":0.96875,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":0.96875,\"pred_targets_consistency_in_neighborhood_mean\":0.9066578474,\"pred_targets_consistency_in_neighborhood_q0\":0.03125,\"pred_targets_consistency_in_neighborhood_q25\":0.90625,\"pred_targets_consistency_in_neighborhood_q50\":0.96875,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":0.96875,\"target_targets_consistency_in_neighborhood_mean\":0.902328034,\"target_targets_consistency_in_neighborhood_q0\":0.03125,\"target_targets_consistency_in_neighborhood_q25\":0.90625,\"target_targets_consistency_in_neighborhood_q50\":0.96875,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.9832561021,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.6666666667,\"targets_and_approxs_consistency_in_neighborhood_q25\":0.96875,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":0.3333333333,\"target_diversity_in_neighborhood_mean\":0.2534039636,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.2006223243,\"target_diversity_in_neighborhood_q75\":0.4488644887,\"target_diversity_in_neighborhood_q1\":1.0,\"target_diversity_in_neighborhood_minmax\":1.0,\"approx_diversity_in_neighborhood_mean\":0.2361448958,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.2006223243,\"approx_diversity_in_neighborhood_q75\":0.4418388624,\"approx_diversity_in_neighborhood_q1\":0.9709505945,\"approx_diversity_in_neighborhood_minmax\":0.9709505944},{\"dataset\":\"wilt\",\"model\":\"nn\",\"attack\":\"pgd\",\"bacc_test\":0.1795180723,\"n_test\":968,\"n_classes\":2,\"neighborhood_size_mean\":1.048553719,\"neighborhood_size_q0\":1.0,\"neighborhood_size_q25\":1.0,\"neighborhood_size_q50\":1.0,\"neighborhood_size_q75\":1.0,\"neighborhood_size_q1\":15.0,\"neighborhood_size_minmax\":14.0,\"uncertainty_mean\":0.0003860825,\"uncertainty_q0\":0.0,\"uncertainty_q25\":0.0,\"uncertainty_q50\":0.0000138439,\"uncertainty_q75\":0.0000231125,\"uncertainty_q1\":0.1718806394,\"uncertainty_minmax\":0.1718806393,\"target_approx_consistency_in_neighborhood_mean\":0.9431818182,\"target_approx_consistency_in_neighborhood_q0\":0.0,\"target_approx_consistency_in_neighborhood_q25\":1.0,\"target_approx_consistency_in_neighborhood_q50\":1.0,\"target_approx_consistency_in_neighborhood_q75\":1.0,\"target_approx_consistency_in_neighborhood_q1\":1.0,\"target_approx_consistency_in_neighborhood_minmax\":1.0,\"pred_targets_consistency_in_neighborhood_mean\":0.3386707989,\"pred_targets_consistency_in_neighborhood_q0\":0.0,\"pred_targets_consistency_in_neighborhood_q25\":0.0,\"pred_targets_consistency_in_neighborhood_q50\":0.0,\"pred_targets_consistency_in_neighborhood_q75\":1.0,\"pred_targets_consistency_in_neighborhood_q1\":1.0,\"pred_targets_consistency_in_neighborhood_minmax\":1.0,\"target_targets_consistency_in_neighborhood_mean\":0.9409435262,\"target_targets_consistency_in_neighborhood_q0\":0.0,\"target_targets_consistency_in_neighborhood_q25\":1.0,\"target_targets_consistency_in_neighborhood_q50\":1.0,\"target_targets_consistency_in_neighborhood_q75\":1.0,\"target_targets_consistency_in_neighborhood_q1\":1.0,\"target_targets_consistency_in_neighborhood_minmax\":1.0,\"targets_and_approxs_consistency_in_neighborhood_mean\":0.997761708,\"targets_and_approxs_consistency_in_neighborhood_q0\":0.0,\"targets_and_approxs_consistency_in_neighborhood_q25\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q50\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q75\":1.0,\"targets_and_approxs_consistency_in_neighborhood_q1\":1.0,\"targets_and_approxs_consistency_in_neighborhood_minmax\":1.0,\"target_diversity_in_neighborhood_mean\":0.0006715108,\"target_diversity_in_neighborhood_q0\":0.0,\"target_diversity_in_neighborhood_q25\":0.0,\"target_diversity_in_neighborhood_q50\":0.0,\"target_diversity_in_neighborhood_q75\":0.0,\"target_diversity_in_neighborhood_q1\":0.6500224216,\"target_diversity_in_neighborhood_minmax\":0.6500224216,\"approx_diversity_in_neighborhood_mean\":0.0,\"approx_diversity_in_neighborhood_q0\":0.0,\"approx_diversity_in_neighborhood_q25\":0.0,\"approx_diversity_in_neighborhood_q50\":0.0,\"approx_diversity_in_neighborhood_q75\":0.0,\"approx_diversity_in_neighborhood_q1\":0.0,\"approx_diversity_in_neighborhood_minmax\":0.0}]"
SUPPORTED_ATTACKS = ["bim", "fgm", "hsj", "org", "pgd", "zoo"]
GRADIENT_ATTACKS = {"bim", "fgm", "pgd"}
BLACKBOX_ATTACKS = {"hsj", "zoo"}
ALLOWED_ATTACKS_BY_MONITORED_MODEL = {
    "nn": ["org", "bim", "fgm", "pgd"],
    "lin": ["org", "hsj", "zoo"],
    "svm": ["org", "hsj", "zoo"],
    "xgb": ["org", "hsj", "zoo"],
}
DIAGNOSTIC_DROP_COLUMNS = [
    "approx",
    "target",
    "pred",
    "error",
    "name",
    "overall_mean_target",
    "scores",
    "mean_target_in_neighborhood",
    "mean_approx_in_neighborhood",
    "neighborhood_size_div_model_avg",
    "neighborhood_size_pct",
    "r_centered_entropy",
    "entropy",
    "logk_r_centered_entropy",
]


def candidate_roots() -> List[Path]:
    roots = [
        KAGGLE_INPUT_DIR,
        KAGGLE_INPUT_DIR.parent,
        Path.cwd(),
        *Path.cwd().parents,
    ]
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists():
        roots.insert(0, kaggle_working)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        roots.extend(p for p in kaggle_input.glob("**") if p.is_dir())
    return roots


def find_results_table() -> Optional[Path]:
    checked = []
    for root in candidate_roots():
        for path in [
            root / "adversarial_upstream" / TARGET_FILE,
            root / "results" / TARGET_FILE,
            root / "data" / "adversarial_upstream" / TARGET_FILE,
            root / "data" / "results" / TARGET_FILE,
            root / TARGET_FILE,
        ]:
            checked.append(path)
            if path.exists():
                return path.resolve()
    print(
        "Could not locate attr_attacks_type_agr_nn.csv. "
        "Using embedded fallback table from the repository snapshot. Checked examples: "
        + ", ".join(str(p) for p in checked[:12])
    )
    return None


def find_optional_file(file_name: str) -> Optional[Path]:
    for root in candidate_roots():
        for path in [
            root / "adversarial_upstream" / file_name,
            root / "results" / file_name,
            root / "data" / "adversarial_upstream" / file_name,
            root / "data" / "results" / file_name,
            root / file_name,
        ]:
            if path.exists():
                return path.resolve()
    return None


TABLE_PATH = find_results_table()
REPO_ROOT = TABLE_PATH.parents[1] if TABLE_PATH is not None and TABLE_PATH.parent.name == "results" else Path.cwd()
RESULTS_DIR = Path("/kaggle/working/results") if Path("/kaggle/working").exists() else REPO_ROOT / "results"
MODEL_DIR = Path("/kaggle/working/models/attack_type_optimized") if Path("/kaggle/working").exists() else REPO_ROOT / "models" / "attack_type_optimized"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TABLE_SOURCE = "file"
if TABLE_PATH is None:
    TABLE_SOURCE = "embedded_fallback"
    TABLE_PATH = RESULTS_DIR / TARGET_FILE
    fallback_df = pd.DataFrame(json.loads(EMBEDDED_ATTACK_TYPE_TABLE_JSON))
    fallback_df.to_csv(TABLE_PATH, index=False)

print(f"TABLE_PATH={TABLE_PATH}")
print(f"TABLE_SOURCE={TABLE_SOURCE}")
print(f"KAGGLE_INPUT_DIR={KAGGLE_INPUT_DIR}")
print(f"RESULTS_DIR={RESULTS_DIR}")
print(f"MODEL_DIR={MODEL_DIR}")
print(f"N_TRIALS_PER_FAMILY={N_TRIALS_PER_FAMILY}")
print(f"SELECTION_MODE={SELECTION_MODE}")
print(f"USE_CONSTRAINED_PREDICTIONS={USE_CONSTRAINED_PREDICTIONS}")
print(f"WINDOWED_AGGREGATION_ENABLED={WINDOWED_AGGREGATION_ENABLED}")

## 2. Load Aggregated Diagnostic Table

In [ ]:
def aggregate_numeric(frame: pd.DataFrame, group_cols: Sequence[str]) -> pd.DataFrame:
    numeric_cols = [
        col for col in frame.columns
        if col not in group_cols
        and col not in DIAGNOSTIC_DROP_COLUMNS
        and pd.api.types.is_numeric_dtype(frame[col])
    ]
    rows = []
    for keys, group in frame.groupby(list(group_cols), dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        for col in numeric_cols:
            values = pd.to_numeric(group[col], errors="coerce").dropna()
            if values.empty:
                continue
            row[f"{col}_mean"] = float(values.mean())
            row[f"{col}_q0"] = float(values.min())
            row[f"{col}_q25"] = float(values.quantile(0.25))
            row[f"{col}_q50"] = float(values.quantile(0.50))
            row[f"{col}_q75"] = float(values.quantile(0.75))
            row[f"{col}_q1"] = float(values.max())
            row[f"{col}_minmax"] = float(values.max() - values.min())
        rows.append(row)
    return pd.DataFrame(rows)


def add_windowed_aggregates(frame: pd.DataFrame, group_cols: Sequence[str]) -> pd.DataFrame:
    full = aggregate_numeric(frame, group_cols)
    full["window_id"] = "full"
    full["window_source"] = "full_group"
    full["window_rows"] = np.nan
    if not WINDOWED_AGGREGATION_ENABLED:
        return full

    rng = np.random.default_rng(RANDOM_STATE)
    window_frames = [full]
    for keys, group in frame.groupby(list(group_cols), dropna=False):
        if len(group) < MIN_WINDOW_ROWS * 2:
            continue
        shuffled = group.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
        chunks = [chunk for chunk in np.array_split(shuffled, WINDOWS_PER_GROUP) if len(chunk) >= MIN_WINDOW_ROWS]
        for idx, chunk in enumerate(chunks):
            agg = aggregate_numeric(chunk, group_cols)
            agg["window_id"] = f"window_{idx}"
            agg["window_source"] = "diagnostic_window"
            agg["window_rows"] = len(chunk)
            window_frames.append(agg)
    return pd.concat(window_frames, ignore_index=True)


def build_training_table() -> Tuple[pd.DataFrame, str]:
    base_diag_path = find_optional_file("attacks_diagnoses.csv")
    nn_diag_path = find_optional_file("attacks_diagnoses_nn.csv")
    if base_diag_path is not None and nn_diag_path is not None:
        print(f"Using raw diagnostics for windowed aggregates: {base_diag_path}, {nn_diag_path}")
        base = pd.read_csv(base_diag_path)
        base = base[(base["dataset"] != "mfeat-morphological") & (base["attack"] != "lpf")].copy()
        base = base[base["attack"].isin(SUPPORTED_ATTACKS)].copy()
        nn_df = pd.read_csv(nn_diag_path)
        nn_df = nn_df[nn_df["attack"].isin(SUPPORTED_ATTACKS)].copy()
        raw = pd.concat([base, nn_df], ignore_index=True)
        table = add_windowed_aggregates(raw, ["dataset", "model", "attack", "bacc_test", "n_test", "n_classes"])
        table.to_csv(RESULTS_DIR / "attr_attacks_type_optimized_windowed.csv", index=False)
        return table, "windowed_raw_diagnostics"

    print("Raw diagnostic CSVs not found. Falling back to the pre-aggregated 39-row table.")
    table = pd.read_csv(TABLE_PATH)
    table["window_id"] = "full"
    table["window_source"] = "preaggregated"
    table["window_rows"] = np.nan
    return table, "preaggregated"


df, TRAINING_TABLE_SOURCE = build_training_table()
df = df[df["attack"].isin(SUPPORTED_ATTACKS)].reset_index(drop=True)

DROP_COLUMNS = ["dataset", "model", "attack", "window_id", "window_source", "window_rows"]
FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in DROP_COLUMNS and pd.api.types.is_numeric_dtype(df[col])
]

encoder = LabelEncoder()
y_all = encoder.fit_transform(df["attack"])
class_labels = list(encoder.classes_)

print("Shape:", df.shape)
print("Training table source:", TRAINING_TABLE_SOURCE)
print("Feature count:", len(FEATURE_COLUMNS))
display(df.head())
display(df["attack"].value_counts().sort_index().rename_axis("attack").reset_index(name="rows"))
display(df.groupby(["dataset", "model", "attack"]).size().unstack(fill_value=0))
display(df["window_source"].value_counts(dropna=False).rename_axis("window_source").reset_index(name="rows"))

## 3. Standalone Training and Evaluation Helpers

In [ ]:
@dataclass
class Candidate:
    family: str
    params: Dict[str, Any]
    variance_threshold: float
    corr_threshold: float
    top_k: Optional[int]
    resampler: str


def adaptive_stratified_folds(y: Sequence[int], max_splits: int = MAX_CV_SPLITS) -> int:
    counts = pd.Series(y).value_counts()
    if counts.empty:
        return 0
    return int(max(0, min(max_splits, counts.min())))


def select_features(
    train_df: pd.DataFrame,
    y_train: np.ndarray,
    feature_cols: Sequence[str],
    variance_threshold: float,
    corr_threshold: float,
    top_k: Optional[int],
) -> List[str]:
    X = train_df[list(feature_cols)].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    variances = X.var(axis=0)
    kept = [col for col in X.columns if float(variances[col]) > variance_threshold]
    if not kept:
        kept = list(feature_cols)

    X_kept = X[kept]
    if corr_threshold < 1.0 and len(kept) > 1:
        corr = X_kept.corr().abs().fillna(0.0)
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]
        kept = [col for col in kept if col not in drop]

    if top_k is not None and len(kept) > top_k:
        try:
            scores = mutual_info_classif(X[kept], y_train, random_state=RANDOM_STATE, discrete_features=False)
            ranked = pd.Series(scores, index=kept).sort_values(ascending=False)
            kept = ranked.head(top_k).index.tolist()
        except Exception:
            kept = kept[:top_k]

    return kept or list(feature_cols)


def clean_matrix(frame: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    return frame[list(cols)].replace([np.inf, -np.inf], np.nan).fillna(0.0)


def guarded_resample(X: pd.DataFrame, y: np.ndarray, strategy: str) -> Tuple[pd.DataFrame, np.ndarray, str]:
    if strategy == "none":
        return X, y, "none"

    counts = pd.Series(y).value_counts()
    if counts.empty or counts.min() < 2:
        if strategy in {"smote", "smoteenn"}:
            strategy = "ros"

    try:
        if strategy == "ros":
            sampler = RandomOverSampler(random_state=RANDOM_STATE)
        elif strategy == "smote":
            sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=1)
        elif strategy == "smoteenn":
            sampler = SMOTEENN(random_state=RANDOM_STATE, smote=SMOTE(random_state=RANDOM_STATE, k_neighbors=1))
        else:
            return X, y, "none"

        X_res, y_res = sampler.fit_resample(X, y)
        if len(np.unique(y_res)) < len(np.unique(y)):
            return X, y, "none"
        return pd.DataFrame(X_res, columns=X.columns), np.asarray(y_res), strategy
    except Exception:
        return X, y, "none"


def make_model(candidate: Candidate, n_classes: int):
    p = dict(candidate.params)
    if candidate.family == "xgb":
        return xgb.XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=n_classes,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
            **p,
        )
    if candidate.family == "catboost":
        return CatBoostClassifier(
            loss_function="MultiClass",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
            **p,
        )
    if candidate.family == "extratrees":
        return ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced", **p)
    if candidate.family == "rf":
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced", **p)
    if candidate.family == "mlp":
        return Pipeline(
            [
                ("scale", StandardScaler()),
                ("mlp", MLPClassifier(random_state=RANDOM_STATE, max_iter=1000, early_stopping=True, **p)),
            ]
        )
    raise ValueError(f"Unknown family: {candidate.family}")


def fit_model(model: Any, X: pd.DataFrame, y: np.ndarray, use_sample_weight: bool = True) -> Any:
    if use_sample_weight and not isinstance(model, Pipeline):
        sample_weight = compute_sample_weight(class_weight="balanced", y=y)
        try:
            model.fit(X, y, sample_weight=sample_weight)
            return model
        except TypeError:
            pass
    model.fit(X, y)
    return model


def predict_labels(model: Any, X: pd.DataFrame) -> np.ndarray:
    pred = model.predict(X)
    pred = np.asarray(pred)
    if pred.ndim == 2 and pred.shape[1] > 1:
        pred = pred.argmax(axis=1)
    return pred.reshape(-1).astype(int)


def allowed_global_labels(monitored_model: str) -> np.ndarray:
    allowed = ALLOWED_ATTACKS_BY_MONITORED_MODEL.get(str(monitored_model), SUPPORTED_ATTACKS)
    labels = [label for label in allowed if label in set(encoder.classes_)]
    return encoder.transform(labels) if labels else np.arange(len(encoder.classes_))


def predict_global_labels(
    model: Any,
    X: pd.DataFrame,
    train_classes: np.ndarray,
    monitored_models: Sequence[str],
) -> np.ndarray:
    pred_local = predict_labels(model, X)
    pred_global = train_classes[np.clip(pred_local, 0, len(train_classes) - 1)]
    if not USE_CONSTRAINED_PREDICTIONS or not hasattr(model, "predict_proba"):
        return pred_global

    try:
        probs = np.asarray(model.predict_proba(X))
    except Exception:
        return pred_global
    if probs.ndim != 2 or probs.shape[1] != len(train_classes):
        return pred_global

    constrained = []
    for row_idx, monitored_model in enumerate(monitored_models):
        allowed = set(allowed_global_labels(str(monitored_model)).tolist())
        local_positions = [pos for pos, global_label in enumerate(train_classes) if int(global_label) in allowed]
        if not local_positions:
            constrained.append(pred_global[row_idx])
            continue
        best_pos = max(local_positions, key=lambda pos: probs[row_idx, pos])
        constrained.append(train_classes[best_pos])
    return np.asarray(constrained, dtype=int)


def metric_row(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {
            "bacc": float(balanced_accuracy_score(y_true, y_pred)),
            "kappa": float(cohen_kappa_score(y_true, y_pred)),
            "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
            "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
            "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        }

In [ ]:
def fit_predict_split(train_df: pd.DataFrame, test_df: pd.DataFrame, candidate: Candidate) -> Tuple[np.ndarray, np.ndarray, List[str], Any, str]:
    y_train_global = encoder.transform(train_df["attack"])
    y_test = encoder.transform(test_df["attack"])
    train_classes = np.asarray(sorted(np.unique(y_train_global)))
    local_map = {label: idx for idx, label in enumerate(train_classes)}
    y_train = np.asarray([local_map[label] for label in y_train_global], dtype=int)
    selected = select_features(
        train_df,
        y_train_global,
        FEATURE_COLUMNS,
        candidate.variance_threshold,
        candidate.corr_threshold,
        candidate.top_k,
    )
    X_train = clean_matrix(train_df, selected)
    X_test = clean_matrix(test_df, selected)
    X_fit, y_fit, actual_resampler = guarded_resample(X_train, y_train, candidate.resampler)
    model = make_model(candidate, n_classes=len(train_classes))
    fit_model(model, X_fit, y_fit, use_sample_weight=(actual_resampler == "none"))
    y_pred = predict_global_labels(model, X_test, train_classes, test_df["model"].tolist())
    return y_test, y_pred, selected, model, actual_resampler


def evaluate_leave_group(candidate: Candidate, group_col: str, scenario: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    pred_rows = []
    for value in sorted(df[group_col].unique()):
        train_df = df[df[group_col] != value].copy()
        test_df = df[df[group_col] == value].copy()
        y_true, y_pred, selected, _model, actual_resampler = fit_predict_split(train_df, test_df, candidate)
        metrics = metric_row(y_true, y_pred)
        metric_rows.append(
            {
                "scenario": scenario,
                "split": value,
                "family": candidate.family,
                "feature_count": len(selected),
                "resampler": actual_resampler,
                **metrics,
            }
        )
        keep_cols = [c for c in ["dataset", "model", "attack", "window_id", "window_source"] if c in test_df.columns]
        out = test_df[keep_cols].copy()
        out["true"] = encoder.inverse_transform(y_true)
        out["pred"] = encoder.inverse_transform(np.clip(y_pred, 0, len(class_labels) - 1))
        out["scenario"] = scenario
        out["split"] = value
        out["family"] = candidate.family
        pred_rows.append(out)
    return pd.DataFrame(metric_rows), pd.concat(pred_rows, ignore_index=True)


def evaluate_cv(candidate: Candidate) -> Tuple[pd.DataFrame, pd.DataFrame]:
    y = encoder.transform(df["attack"])
    n_splits = adaptive_stratified_folds(y)
    if n_splits < 2:
        return pd.DataFrame(), pd.DataFrame()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    metric_rows = []
    pred_rows = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(df[FEATURE_COLUMNS], y)):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        y_true, y_pred, selected, _model, actual_resampler = fit_predict_split(train_df, test_df, candidate)
        metric_rows.append(
            {
                "scenario": "10-fold cross-validation",
                "split": fold,
                "family": candidate.family,
                "feature_count": len(selected),
                "resampler": actual_resampler,
                **metric_row(y_true, y_pred),
            }
        )
        keep_cols = [c for c in ["dataset", "model", "attack", "window_id", "window_source"] if c in test_df.columns]
        out = test_df[keep_cols].copy()
        out["true"] = encoder.inverse_transform(y_true)
        out["pred"] = encoder.inverse_transform(np.clip(y_pred, 0, len(class_labels) - 1))
        out["scenario"] = "10-fold cross-validation"
        out["split"] = fold
        out["family"] = candidate.family
        pred_rows.append(out)
    return pd.DataFrame(metric_rows), pd.concat(pred_rows, ignore_index=True)


def evaluate_candidate(candidate: Candidate, include_cv: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    metrics = []
    predictions = []
    m, p = evaluate_leave_group(candidate, "dataset", "one-data-set-out")
    metrics.append(m)
    predictions.append(p)
    m, p = evaluate_leave_group(candidate, "model", "one-model-out")
    metrics.append(m)
    predictions.append(p)
    if include_cv:
        m, p = evaluate_cv(candidate)
        if not m.empty:
            metrics.append(m)
            predictions.append(p)
    return pd.concat(metrics, ignore_index=True), pd.concat(predictions, ignore_index=True)


def selection_scores(candidate: Candidate) -> Dict[str, float]:
    dataset_metrics, _ = evaluate_leave_group(candidate, "dataset", "one-data-set-out")
    model_metrics, _ = evaluate_leave_group(candidate, "model", "one-model-out")
    dataset_bacc = float(dataset_metrics["bacc"].mean())
    dataset_kappa = float(dataset_metrics["kappa"].mean())
    model_bacc = float(model_metrics["bacc"].mean())
    model_kappa = float(model_metrics["kappa"].mean())
    if SELECTION_MODE == "dataset":
        combined_bacc = dataset_bacc
        combined_kappa = dataset_kappa
    else:
        combined_bacc = DATASET_SCORE_WEIGHT * dataset_bacc + MODEL_SCORE_WEIGHT * model_bacc
        combined_kappa = DATASET_SCORE_WEIGHT * dataset_kappa + MODEL_SCORE_WEIGHT * model_kappa
    return {
        "dataset_bacc": dataset_bacc,
        "dataset_kappa": dataset_kappa,
        "model_bacc": model_bacc,
        "model_kappa": model_kappa,
        "selection_bacc": combined_bacc,
        "selection_kappa": combined_kappa,
    }

## 4. Optuna Search

In [ ]:
def sample_common(trial: optuna.Trial) -> Dict[str, Any]:
    top_k_choice = trial.suggest_categorical("top_k", ["none", 12, 20, 35, 50])
    return {
        "variance_threshold": trial.suggest_categorical("variance_threshold", [0.0, 1e-10, 1e-6, 1e-4]),
        "corr_threshold": trial.suggest_categorical("corr_threshold", [0.90, 0.95, 0.98, 1.0]),
        "top_k": None if top_k_choice == "none" else int(top_k_choice),
        "resampler": trial.suggest_categorical("resampler", ["none", "ros", "smote", "smoteenn"]),
    }


def sample_candidate(trial: optuna.Trial, family: str) -> Candidate:
    common = sample_common(trial)
    if family == "xgb":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 80, 800, step=40),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.35, log=True),
            "subsample": trial.suggest_float("subsample", 0.55, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 8.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 2.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 8.0, log=True),
        }
    elif family == "catboost":
        params = {
            "iterations": trial.suggest_int("iterations", 80, 700, step=40),
            "depth": trial.suggest_int("depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.35, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.0, 4.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 3.0),
        }
    elif family == "extratrees":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 900, step=50),
            "max_depth": trial.suggest_categorical("max_depth", [None, 3, 5, 8, 12, 20]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        }
    elif family == "rf":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 900, step=50),
            "max_depth": trial.suggest_categorical("max_depth", [None, 3, 5, 8, 12, 20, 50]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        }
    elif family == "mlp":
        params = {
            "hidden_layer_sizes": trial.suggest_categorical("hidden_layer_sizes", [(16,), (32,), (32, 16), (64, 32)]),
            "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
            "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 5e-2, log=True),
            "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
        }
    else:
        raise ValueError(family)
    return Candidate(family=family, params=params, **common)


def optimize_family(family: str, n_trials: int = N_TRIALS_PER_FAMILY) -> Tuple[Candidate, pd.DataFrame]:
    def objective(trial: optuna.Trial) -> float:
        candidate = sample_candidate(trial, family)
        trial.set_user_attr("candidate", {
            "family": candidate.family,
            "params": candidate.params,
            "variance_threshold": candidate.variance_threshold,
            "corr_threshold": candidate.corr_threshold,
            "top_k": candidate.top_k,
            "resampler": candidate.resampler,
        })
        try:
            scores = selection_scores(candidate)
        except Exception as exc:
            trial.set_user_attr("error", repr(exc))
            return -999.0
        for key, value in scores.items():
            trial.set_user_attr(key, value)
        return scores["selection_bacc"] + 0.001 * scores["selection_kappa"]

    study = optuna.create_study(direction="maximize", study_name=f"attack_type_{family}")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best_attrs = study.best_trial.user_attrs["candidate"]
    best = Candidate(
        family=best_attrs["family"],
        params=best_attrs["params"],
        variance_threshold=best_attrs["variance_threshold"],
        corr_threshold=best_attrs["corr_threshold"],
        top_k=best_attrs["top_k"],
        resampler=best_attrs["resampler"],
    )
    rows = []
    for t in study.trials:
        row = {
            "family": family,
            "trial": t.number,
            "objective": t.value,
            "selection_bacc": t.user_attrs.get("selection_bacc", np.nan),
            "selection_kappa": t.user_attrs.get("selection_kappa", np.nan),
            "dataset_bacc": t.user_attrs.get("dataset_bacc", np.nan),
            "dataset_kappa": t.user_attrs.get("dataset_kappa", np.nan),
            "model_bacc": t.user_attrs.get("model_bacc", np.nan),
            "model_kappa": t.user_attrs.get("model_kappa", np.nan),
            "state": str(t.state),
        }
        row.update(t.params)
        rows.append(row)
    return best, pd.DataFrame(rows)


families = ["xgb", "catboost", "extratrees", "rf", "mlp"]
best_candidates: Dict[str, Candidate] = {}
trial_frames = []

for family in families:
    print(f"Optimizing {family}...")
    best, trials = optimize_family(family)
    best_candidates[family] = best
    trial_frames.append(trials)
    print(best)
    display(trials.sort_values(["selection_bacc", "selection_kappa"], ascending=False).head(5))

trials_all = pd.concat(trial_frames, ignore_index=True)
trials_path = RESULTS_DIR / "attack_type_optimized_trials.csv"
trials_all.to_csv(trials_path, index=False)
print(trials_path)

## 5. Soft-Voting Candidate

In [ ]:
def fit_final_candidate(candidate: Candidate, train_df: pd.DataFrame) -> Tuple[Any, List[str], str]:
    y_train = encoder.transform(train_df["attack"])
    selected = select_features(
        train_df,
        y_train,
        FEATURE_COLUMNS,
        candidate.variance_threshold,
        candidate.corr_threshold,
        candidate.top_k,
    )
    X_train = clean_matrix(train_df, selected)
    X_fit, y_fit, actual_resampler = guarded_resample(X_train, y_train, candidate.resampler)
    model = make_model(candidate, n_classes=len(class_labels))
    fit_model(model, X_fit, y_fit, use_sample_weight=(actual_resampler == "none"))
    return model, selected, actual_resampler


def fit_final_group_candidate(candidate: Candidate, train_df: pd.DataFrame) -> Tuple[Any, LabelEncoder, List[str], str]:
    group_encoder = LabelEncoder()
    y_group = group_encoder.fit_transform(train_df["attack"].map(attack_group))
    selected = select_features(
        train_df,
        y_group,
        FEATURE_COLUMNS,
        candidate.variance_threshold,
        candidate.corr_threshold,
        candidate.top_k,
    )
    X_train = clean_matrix(train_df, selected)
    X_fit, y_fit, actual_resampler = guarded_resample(X_train, y_group, candidate.resampler)
    model = make_model(candidate, n_classes=len(group_encoder.classes_))
    fit_model(model, X_fit, y_fit, use_sample_weight=(actual_resampler == "none"))
    return model, group_encoder, selected, actual_resampler


def make_voting_candidate(base_candidates: Dict[str, Candidate]) -> Candidate:
    return Candidate(
        family="voting",
        params={"members": ["xgb", "catboost", "extratrees"]},
        variance_threshold=0.0,
        corr_threshold=0.98,
        top_k=None,
        resampler="none",
    )


def fit_predict_voting_split(train_df: pd.DataFrame, test_df: pd.DataFrame, members: Sequence[str]) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    y_train_global = encoder.transform(train_df["attack"])
    y_test = encoder.transform(test_df["attack"])
    train_classes = np.asarray(sorted(np.unique(y_train_global)))
    local_map = {label: idx for idx, label in enumerate(train_classes)}
    y_train = np.asarray([local_map[label] for label in y_train_global], dtype=int)
    selected_sets = []
    estimators = []
    for member in members:
        candidate = best_candidates[member]
        selected = select_features(
            train_df,
            y_train_global,
            FEATURE_COLUMNS,
            candidate.variance_threshold,
            candidate.corr_threshold,
            candidate.top_k,
        )
        selected_sets.append(set(selected))
    selected = sorted(set.intersection(*selected_sets)) if selected_sets else list(FEATURE_COLUMNS)
    if not selected:
        selected = list(FEATURE_COLUMNS)
    X_train = clean_matrix(train_df, selected)
    X_test = clean_matrix(test_df, selected)
    X_fit, y_fit, _actual_resampler = guarded_resample(X_train, y_train, "ros")
    for member in members:
        candidate = best_candidates[member]
        estimator = make_model(Candidate(member, candidate.params, 0.0, 1.0, None, "none"), len(train_classes))
        estimators.append((member, estimator))
    voter = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
    voter.fit(X_fit, y_fit)
    y_pred = predict_global_labels(voter, X_test, train_classes, test_df["model"].tolist())
    return y_test, y_pred, selected


def evaluate_voting(members: Sequence[str] = ("xgb", "catboost", "extratrees")) -> Tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    pred_rows = []
    for group_col, scenario in [("dataset", "one-data-set-out"), ("model", "one-model-out")]:
        for value in sorted(df[group_col].unique()):
            train_df = df[df[group_col] != value].copy()
            test_df = df[df[group_col] == value].copy()
            y_true, y_pred, selected = fit_predict_voting_split(train_df, test_df, members)
            metric_rows.append(
                {
                    "scenario": scenario,
                    "split": value,
                    "family": "voting",
                    "feature_count": len(selected),
                    "resampler": "ros",
                    **metric_row(y_true, y_pred),
                }
            )
            keep_cols = [c for c in ["dataset", "model", "attack", "window_id", "window_source"] if c in test_df.columns]
            out = test_df[keep_cols].copy()
            out["true"] = encoder.inverse_transform(y_true)
            out["pred"] = encoder.inverse_transform(np.clip(y_pred, 0, len(class_labels) - 1))
            out["scenario"] = scenario
            out["split"] = value
            out["family"] = "voting"
            pred_rows.append(out)
    return pd.DataFrame(metric_rows), pd.concat(pred_rows, ignore_index=True)


voting_candidate = make_voting_candidate(best_candidates)
voting_metrics, voting_predictions = evaluate_voting()
display(voting_metrics)

## 6. Evaluate Best Candidates and Save Reports

In [ ]:
metric_frames = []
prediction_frames = []
candidate_records = {}

for family, candidate in best_candidates.items():
    print(f"Evaluating best {family}...")
    metrics, predictions = evaluate_candidate(candidate, include_cv=True)
    metric_frames.append(metrics)
    prediction_frames.append(predictions)
    dataset_primary = metrics[metrics["scenario"] == "one-data-set-out"]
    model_primary = metrics[metrics["scenario"] == "one-model-out"]
    dataset_bacc = float(dataset_primary["bacc"].mean())
    dataset_kappa = float(dataset_primary["kappa"].mean())
    model_bacc = float(model_primary["bacc"].mean())
    model_kappa = float(model_primary["kappa"].mean())
    candidate_records[family] = {
        "candidate": candidate,
        "dataset_bacc": dataset_bacc,
        "dataset_kappa": dataset_kappa,
        "model_bacc": model_bacc,
        "model_kappa": model_kappa,
        "selection_bacc": DATASET_SCORE_WEIGHT * dataset_bacc + MODEL_SCORE_WEIGHT * model_bacc,
        "selection_kappa": DATASET_SCORE_WEIGHT * dataset_kappa + MODEL_SCORE_WEIGHT * model_kappa,
    }

metric_frames.append(voting_metrics)
prediction_frames.append(voting_predictions)
dataset_voting = voting_metrics[voting_metrics["scenario"] == "one-data-set-out"]
model_voting = voting_metrics[voting_metrics["scenario"] == "one-model-out"]
dataset_bacc = float(dataset_voting["bacc"].mean())
dataset_kappa = float(dataset_voting["kappa"].mean())
model_bacc = float(model_voting["bacc"].mean())
model_kappa = float(model_voting["kappa"].mean())
candidate_records["voting"] = {
    "candidate": voting_candidate,
    "dataset_bacc": dataset_bacc,
    "dataset_kappa": dataset_kappa,
    "model_bacc": model_bacc,
    "model_kappa": model_kappa,
    "selection_bacc": DATASET_SCORE_WEIGHT * dataset_bacc + MODEL_SCORE_WEIGHT * model_bacc,
    "selection_kappa": DATASET_SCORE_WEIGHT * dataset_kappa + MODEL_SCORE_WEIGHT * model_kappa,
}

metrics_all = pd.concat(metric_frames, ignore_index=True)
predictions_all = pd.concat(prediction_frames, ignore_index=True)

leaderboard = (
    pd.DataFrame(
        [
            {
                "family": family,
                "selection_bacc": rec["selection_bacc"],
                "selection_kappa": rec["selection_kappa"],
                "dataset_bacc": rec["dataset_bacc"],
                "dataset_kappa": rec["dataset_kappa"],
                "model_bacc": rec["model_bacc"],
                "model_kappa": rec["model_kappa"],
            }
            for family, rec in candidate_records.items()
        ]
    )
    .sort_values(["selection_bacc", "selection_kappa"], ascending=False)
    .reset_index(drop=True)
)

display(leaderboard)
display(metrics_all.groupby(["scenario", "family"])[["bacc", "kappa", "precision_weighted", "recall_weighted", "f1_weighted"]].agg(["mean", "std"]).round(4))

In [ ]:
def save_confusion_matrices(predictions: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    rows = []
    labels = class_labels
    for (scenario, family), group in predictions.groupby(["scenario", "family"]):
        cm = confusion_matrix(group["true"], group["pred"], labels=labels)
        for i, true_label in enumerate(labels):
            for j, pred_label in enumerate(labels):
                rows.append(
                    {
                        "scenario": scenario,
                        "family": family,
                        "true": true_label,
                        "pred": pred_label,
                        "count": int(cm[i, j]),
                    }
                )
    out = pd.DataFrame(rows)
    out.to_csv(out_path, index=False)
    return out


def per_class_report(predictions: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    rows = []
    for (scenario, family), group in predictions.groupby(["scenario", "family"]):
        report = classification_report(group["true"], group["pred"], labels=class_labels, output_dict=True, zero_division=0)
        for label in class_labels:
            rows.append(
                {
                    "scenario": scenario,
                    "family": family,
                    "label": label,
                    "precision": report[label]["precision"],
                    "recall": report[label]["recall"],
                    "f1": report[label]["f1-score"],
                    "support": report[label]["support"],
                }
            )
    out = pd.DataFrame(rows)
    out.to_csv(out_path, index=False)
    return out


def attack_group(label: str) -> str:
    if label in GRADIENT_ATTACKS:
        return "gradient_attack"
    if label in BLACKBOX_ATTACKS:
        return "blackbox_attack"
    return "org"


def grouped_attack_metrics(predictions: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    rows = []
    grouped = predictions.copy()
    grouped["true_group"] = grouped["true"].map(attack_group)
    grouped["pred_group"] = grouped["pred"].map(attack_group)
    for (scenario, family), group in grouped.groupby(["scenario", "family"]):
        y_true = group["true_group"]
        y_pred = group["pred_group"]
        rows.append(
            {
                "scenario": scenario,
                "family": family,
                "bacc": balanced_accuracy_score(y_true, y_pred),
                "kappa": cohen_kappa_score(y_true, y_pred),
                "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
                "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
                "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            }
        )
    out = pd.DataFrame(rows)
    out.to_csv(out_path, index=False)
    return out


metrics_path = RESULTS_DIR / "attack_type_optimized_metrics.csv"
predictions_path = RESULTS_DIR / "attack_type_optimized_predictions.csv"
confusion_path = RESULTS_DIR / "attack_type_optimized_confusion_matrices.csv"
per_class_path = RESULTS_DIR / "attack_type_optimized_per_class_metrics.csv"
grouped_metrics_path = RESULTS_DIR / "attack_type_optimized_grouped_metrics.csv"
best_params_path = RESULTS_DIR / "attack_type_optimized_best_params.json"

metrics_all.to_csv(metrics_path, index=False)
predictions_all.to_csv(predictions_path, index=False)
confusion_df = save_confusion_matrices(predictions_all, confusion_path)
per_class_df = per_class_report(predictions_all, per_class_path)
grouped_metrics_df = grouped_attack_metrics(predictions_all, grouped_metrics_path)

best_family = leaderboard.iloc[0]["family"]
best_record = candidate_records[best_family]
best_candidate = best_record["candidate"]

best_params_payload = {
    "selected_family": best_family,
    "selection_mode": SELECTION_MODE,
    "dataset_score_weight": DATASET_SCORE_WEIGHT,
    "model_score_weight": MODEL_SCORE_WEIGHT,
    "selection_bacc": best_record["selection_bacc"],
    "selection_kappa": best_record["selection_kappa"],
    "dataset_bacc": best_record["dataset_bacc"],
    "dataset_kappa": best_record["dataset_kappa"],
    "model_bacc": best_record["model_bacc"],
    "model_kappa": best_record["model_kappa"],
    "leaderboard": leaderboard.to_dict(orient="records"),
    "candidates": {
        family: {
            "family": rec["candidate"].family,
            "params": rec["candidate"].params,
            "variance_threshold": rec["candidate"].variance_threshold,
            "corr_threshold": rec["candidate"].corr_threshold,
            "top_k": rec["candidate"].top_k,
            "resampler": rec["candidate"].resampler,
            "selection_bacc": rec["selection_bacc"],
            "selection_kappa": rec["selection_kappa"],
            "dataset_bacc": rec["dataset_bacc"],
            "dataset_kappa": rec["dataset_kappa"],
            "model_bacc": rec["model_bacc"],
            "model_kappa": rec["model_kappa"],
        }
        for family, rec in candidate_records.items()
    },
}
best_params_path.write_text(json.dumps(best_params_payload, indent=2), encoding="utf-8")

print("Saved:")
for path in [metrics_path, predictions_path, confusion_path, per_class_path, grouped_metrics_path, trials_path, best_params_path]:
    print(path)

## 7. Fit and Save Final Deployable Model

In [ ]:
def final_feature_importance(model: Any, feature_cols: Sequence[str], family: str) -> pd.DataFrame:
    if family == "voting":
        rows = []
        for name, estimator in model.named_estimators_.items():
            importance = getattr(estimator, "feature_importances_", None)
            if importance is not None:
                rows.append(pd.DataFrame({"family": name, "var": list(feature_cols), "importance": importance}))
        if rows:
            out = pd.concat(rows, ignore_index=True)
            avg = out.groupby("var", as_index=False)["importance"].mean()
            avg["family"] = "voting_mean"
            avg["rank"] = avg["importance"].rank(ascending=False)
            return avg.sort_values("rank")
        return pd.DataFrame(columns=["family", "var", "importance", "rank"])
    raw_model = model
    if isinstance(model, Pipeline):
        raw_model = model.steps[-1][1]
    importance = getattr(raw_model, "feature_importances_", None)
    if importance is None:
        return pd.DataFrame(columns=["family", "var", "importance", "rank"])
    out = pd.DataFrame({"family": family, "var": list(feature_cols), "importance": importance})
    out["rank"] = out["importance"].rank(ascending=False)
    return out.sort_values("rank")


if best_family == "voting":
    members = best_candidate.params["members"]
    selected_sets = []
    y_full = encoder.transform(df["attack"])
    for member in members:
        candidate = best_candidates[member]
        selected_sets.append(
            set(
                select_features(
                    df,
                    y_full,
                    FEATURE_COLUMNS,
                    candidate.variance_threshold,
                    candidate.corr_threshold,
                    candidate.top_k,
                )
            )
        )
    final_features = sorted(set.intersection(*selected_sets)) if selected_sets else list(FEATURE_COLUMNS)
    if not final_features:
        final_features = list(FEATURE_COLUMNS)
    X_full = clean_matrix(df, final_features)
    X_fit, y_fit, final_resampler = guarded_resample(X_full, y_full, "ros")
    estimators = []
    for member in members:
        candidate = best_candidates[member]
        estimators.append((member, make_model(Candidate(member, candidate.params, 0.0, 1.0, None, "none"), len(class_labels))))
    final_model = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
    final_model.fit(X_fit, y_fit)
else:
    final_model, final_features, final_resampler = fit_final_candidate(best_candidate, df)

model_path = MODEL_DIR / "attack_type_optimized_best.joblib"
encoder_path = MODEL_DIR / "attack_type_optimized_label_encoder.joblib"
features_path = MODEL_DIR / "attack_type_optimized_features.json"
config_path = MODEL_DIR / "attack_type_optimized_config.json"
family_model_path = MODEL_DIR / "attack_family_optimized_best.joblib"
family_encoder_path = MODEL_DIR / "attack_family_optimized_label_encoder.joblib"
family_features_path = MODEL_DIR / "attack_family_optimized_features.json"
family_config_path = MODEL_DIR / "attack_family_optimized_config.json"
fi_path = RESULTS_DIR / "attack_type_optimized_feature_importance.csv"

joblib.dump(final_model, model_path)
joblib.dump(encoder, encoder_path)
features_path.write_text(json.dumps(final_features, indent=2), encoding="utf-8")

family_model, family_encoder, family_features, family_resampler = fit_final_group_candidate(best_candidate, df)
joblib.dump(family_model, family_model_path)
joblib.dump(family_encoder, family_encoder_path)
family_features_path.write_text(json.dumps(family_features, indent=2), encoding="utf-8")

config = {
    "task": "attack_type",
    "selected_family": best_family,
    "params": best_candidate.params,
    "feature_columns": final_features,
    "preprocessing": {
        "variance_threshold": best_candidate.variance_threshold,
        "corr_threshold": best_candidate.corr_threshold,
        "top_k": best_candidate.top_k,
    },
    "resampling_strategy_requested": best_candidate.resampler,
    "resampling_strategy_final": final_resampler,
    "classes": class_labels,
    "attack_groups": {
        "gradient_attack": sorted(GRADIENT_ATTACKS),
        "blackbox_attack": sorted(BLACKBOX_ATTACKS),
        "org": ["org"],
    },
    "prediction_constraints_enabled": USE_CONSTRAINED_PREDICTIONS,
    "allowed_attacks_by_monitored_model": ALLOWED_ATTACKS_BY_MONITORED_MODEL,
    "selection_mode": SELECTION_MODE,
    "dataset_score_weight": DATASET_SCORE_WEIGHT,
    "model_score_weight": MODEL_SCORE_WEIGHT,
    "selection_bacc": best_record["selection_bacc"],
    "selection_kappa": best_record["selection_kappa"],
    "dataset_bacc": best_record["dataset_bacc"],
    "dataset_kappa": best_record["dataset_kappa"],
    "model_bacc": best_record["model_bacc"],
    "model_kappa": best_record["model_kappa"],
    "training_table_source": TRAINING_TABLE_SOURCE,
    "windowed_aggregation_enabled": WINDOWED_AGGREGATION_ENABLED,
    "notes": "Experimental optimized attack-type classifier. Current table is very small; validate on new diagnostic rows before relying on it.",
}
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

family_config = {
    "task": "attack_family",
    "selected_family": best_family,
    "params": best_candidate.params,
    "feature_columns": family_features,
    "preprocessing": {
        "variance_threshold": best_candidate.variance_threshold,
        "corr_threshold": best_candidate.corr_threshold,
        "top_k": best_candidate.top_k,
    },
    "resampling_strategy_requested": best_candidate.resampler,
    "resampling_strategy_final": family_resampler,
    "classes": list(family_encoder.classes_),
    "attack_groups": {
        "gradient_attack": sorted(GRADIENT_ATTACKS),
        "blackbox_attack": sorted(BLACKBOX_ATTACKS),
        "org": ["org"],
    },
    "selection_source": "same candidate family/params selected by exact attack-type combined score",
    "training_table_source": TRAINING_TABLE_SOURCE,
    "windowed_aggregation_enabled": WINDOWED_AGGREGATION_ENABLED,
    "notes": "Direct grouped attack-family classifier. Intended for demo/report use when exact bim/fgm/pgd separation is unstable.",
}
family_config_path.write_text(json.dumps(family_config, indent=2), encoding="utf-8")

fi = final_feature_importance(final_model, final_features, best_family)
fi.to_csv(fi_path, index=False)

print("Saved final optimized attack-type artifacts:")
for path in [
    model_path,
    encoder_path,
    features_path,
    config_path,
    family_model_path,
    family_encoder_path,
    family_features_path,
    family_config_path,
    fi_path,
]:
    print(path)
display(fi.head(20))

## 8. Reload Smoke Test

In [ ]:
loaded_model = joblib.load(model_path)
loaded_encoder = joblib.load(encoder_path)
loaded_features = json.loads(features_path.read_text(encoding="utf-8"))
sample_X = clean_matrix(df.iloc[[0]], loaded_features)
pred_enc = predict_labels(loaded_model, sample_X)
pred_label = loaded_encoder.inverse_transform(np.clip(pred_enc, 0, len(loaded_encoder.classes_) - 1))[0]

print("Reload test prediction:", pred_label)
print("Allowed labels:", list(loaded_encoder.classes_))
assert pred_label in set(loaded_encoder.classes_)

loaded_family_model = joblib.load(family_model_path)
loaded_family_encoder = joblib.load(family_encoder_path)
loaded_family_features = json.loads(family_features_path.read_text(encoding="utf-8"))
family_X = clean_matrix(df.iloc[[0]], loaded_family_features)
family_pred_enc = predict_labels(loaded_family_model, family_X)
family_pred_label = loaded_family_encoder.inverse_transform(
    np.clip(family_pred_enc, 0, len(loaded_family_encoder.classes_) - 1)
)[0]

print("Reload family prediction:", family_pred_label)
print("Allowed family labels:", list(loaded_family_encoder.classes_))
assert family_pred_label in set(loaded_family_encoder.classes_)

## 9. Summary

In [ ]:
summary = metrics_all.groupby(["scenario", "family"])[["bacc", "kappa", "f1_weighted"]].agg(["mean", "std"]).round(4)
group_summary = grouped_metrics_df.groupby(["scenario", "family"])[["bacc", "kappa", "f1_weighted"]].mean().round(4)
print("Current baseline reference from previous run:")
print("  RF leave-one-dataset-out mean bacc ~= 0.3704")
print("  XGB leave-one-dataset-out mean bacc ~= 0.3935")
print()
print("Optimized leaderboard:")
display(leaderboard)
display(summary)
print("Grouped attack labels: gradient_attack=(bim, fgm, pgd), blackbox_attack=(hsj, zoo), org=org")
display(group_summary)